# Digi prodej vstvy

In [0]:

-- CREATE SCHEMA IF NOT EXISTS vse_banka.digi_prodej_uc_v5;

## Bronze

In [0]:


CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_ads_campaign_performance
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_ads_campaign_performance.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_web_sessions
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_web_sessions.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_web_conversions
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_web_conversions.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_identity_map
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_identity_map.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_plan_digital_clients
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_plan_digital_clients.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_plan_digital_accounts
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_plan_digital_accounts.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_crm_clients
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_crm_clients.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_crm_accounts
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_crm_accounts.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_transactions
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/bronze_transactions.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_team1_digi_income_cost_current_2025
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/team1-digi_income_cost_current_2025.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_team1_digi_income_cost_2026
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/team1-digi_income_cost_2026.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_plan_acquisition_monthly
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/demo/plan_acquisition_monthly.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_plan_insurance_targets
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/demo/plan_insurance_targets.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.bronze_plan_lending_targets
AS SELECT * FROM read_files(
  '/Volumes/vse_banka/digi_prodej/bronze_data/demo/plan_lending_targets.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.external_customer_service_customer AS
SELECT * FROM vse_banka.obsluha_klienta.silver_customer;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.external_customer_service_address AS
SELECT * FROM vse_banka.obsluha_klienta.silver_address;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.external_customer_service_customer_consent AS
SELECT * FROM vse_banka.obsluha_klienta.silver_customer_consent;

## Silver

In [0]:


CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_web_sessions AS
SELECT
  session_id,
  user_id,
  CAST(event_time AS TIMESTAMP) AS event_time,
  LOWER(source) AS source,
  LOWER(medium) AS medium,
  LOWER(campaign) AS campaign,
  LOWER(device_type) AS device_type,
  UPPER(region) AS region,
  gclid,
  fbclid,
  landing_page,
  dq_flag
FROM vse_banka.digi_prodej.bronze_web_sessions
WHERE dq_flag <> 'FAIL';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_web_conversions AS
SELECT
  conversion_id,
  user_id,
  session_id,
  CAST(event_time AS TIMESTAMP) AS event_time,
  LOWER(conversion_type) AS conversion_type,
  form_id,
  dq_flag
FROM vse_banka.digi_prodej.bronze_web_conversions
WHERE dq_flag <> 'FAIL'
  AND LOWER(conversion_type) = 'account_application';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_identity_map AS
SELECT user_id, hashed_email, client_id, dq_flag
FROM vse_banka.digi_prodej.bronze_identity_map
WHERE dq_flag <> 'FAIL';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_customer_profile_link AS
WITH internal_clients AS (
  SELECT client_id, 'current' AS client_scope, ROW_NUMBER() OVER (ORDER BY client_id) AS rn
  FROM vse_banka.digi_prodej.bronze_crm_clients
  UNION ALL
  SELECT client_id, 'plan' AS client_scope, ROW_NUMBER() OVER (ORDER BY client_id) + 1000000 AS rn
  FROM vse_banka.digi_prodej.bronze_plan_digital_clients
),
external_customers AS (
  SELECT
    CAST(customer_id AS STRING) AS external_customer_id,
    ROW_NUMBER() OVER (ORDER BY customer_id) AS rn,
    COUNT(*) OVER () AS cnt
  FROM vse_banka.digi_prodej.external_customer_service_customer
),
external_addresses AS (
  SELECT
    CAST(address_id AS STRING) AS external_address_id,
    ROW_NUMBER() OVER (ORDER BY address_id) AS rn,
    COUNT(*) OVER () AS cnt
  FROM vse_banka.digi_prodej.external_customer_service_address
),
seeded AS (
  SELECT
    i.client_id,
    i.client_scope,
    c.external_customer_id,
    a.external_address_id,
    CASE
      WHEN PMOD(xxhash64(CONCAT(i.client_id, '_band')), 100) < 28 THEN 18 + PMOD(xxhash64(CONCAT(i.client_id, '_a1')), 7)
      WHEN PMOD(xxhash64(CONCAT(i.client_id, '_band')), 100) < 59 THEN 25 + PMOD(xxhash64(CONCAT(i.client_id, '_a2')), 10)
      WHEN PMOD(xxhash64(CONCAT(i.client_id, '_band')), 100) < 78 THEN 35 + PMOD(xxhash64(CONCAT(i.client_id, '_a3')), 10)
      WHEN PMOD(xxhash64(CONCAT(i.client_id, '_band')), 100) < 91 THEN 45 + PMOD(xxhash64(CONCAT(i.client_id, '_a4')), 10)
      ELSE 55 + PMOD(xxhash64(CONCAT(i.client_id, '_a5')), 16)
    END AS synthetic_age_years
  FROM internal_clients i
  JOIN external_customers c
    ON PMOD(i.rn - 1, c.cnt) + 1 = c.rn
  JOIN external_addresses a
    ON PMOD(i.rn - 1, a.cnt) + 1 = a.rn
)
SELECT
  client_id,
  client_scope,
  external_customer_id,
  external_address_id,
  DATE_SUB(
    ADD_MONTHS(DATE('2025-12-31'), -12 * synthetic_age_years),
    CAST(PMOD(xxhash64(CONCAT(client_id, '_day')), 365) AS INT)
  ) AS birth_date,
  'PASS' AS dq_flag
FROM seeded;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_customer_profile_enriched AS
WITH active_marketing_consent AS (
  SELECT
    CAST(customer_id AS STRING) AS external_customer_id,
    MAX(CASE WHEN UPPER(consent_channel) = 'PUSH' THEN 1 ELSE 0 END) AS consent_push,
    MAX(CASE WHEN UPPER(consent_channel) = 'EMAIL' THEN 1 ELSE 0 END) AS consent_email,
    MAX(CASE WHEN UPPER(consent_channel) = 'SMS' THEN 1 ELSE 0 END) AS consent_sms,
    MAX(CASE WHEN UPPER(consent_channel) = 'PHONE' THEN 1 ELSE 0 END) AS consent_phone,
    MAX(1) AS consent_any_marketing
  FROM vse_banka.digi_prodej.external_customer_service_customer_consent
  WHERE UPPER(consent_type) = 'MARKETING'
    --AND CAST(is_granted AS BOOLEAN) = TRUE
    --AND CURRENT_DATE() BETWEEN DATE(valid_from) AND DATE(valid_to)
  GROUP BY CAST(customer_id AS STRING)
)
SELECT
  l.client_id,
  l.client_scope,
  l.external_customer_id,
  l.external_address_id,
  l.birth_date,
  CAST(FLOOR(DATEDIFF(DATE('2025-12-31'), l.birth_date) / 365.25) AS INT) AS age_years,
  CASE
    WHEN FLOOR(DATEDIFF(DATE('2025-12-31'), l.birth_date) / 365.25) BETWEEN 18 AND 24 THEN '18-24'
    WHEN FLOOR(DATEDIFF(DATE('2025-12-31'), l.birth_date) / 365.25) BETWEEN 25 AND 34 THEN '25-34'
    WHEN FLOOR(DATEDIFF(DATE('2025-12-31'), l.birth_date) / 365.25) BETWEEN 35 AND 44 THEN '35-44'
    WHEN FLOOR(DATEDIFF(DATE('2025-12-31'), l.birth_date) / 365.25) BETWEEN 45 AND 54 THEN '45-54'
    ELSE '55+'
  END AS age_band,
  UPPER(COALESCE(a.region, 'CZ')) AS region,
  c.customer_segment,
  c.source_system,
  CAST(COALESCE(mc.consent_any_marketing, 0) AS BOOLEAN) AS consent_any_marketing,
  CAST(COALESCE(mc.consent_push, 0) AS BOOLEAN) AS consent_push,
  CAST(COALESCE(mc.consent_email, 0) AS BOOLEAN) AS consent_email,
  CAST(COALESCE(mc.consent_sms, 0) AS BOOLEAN) AS consent_sms,
  CAST(COALESCE(mc.consent_phone, 0) AS BOOLEAN) AS consent_phone,
  l.dq_flag
FROM vse_banka.digi_prodej.silver_customer_profile_link l
LEFT JOIN vse_banka.digi_prodej.external_customer_service_customer c
  ON CAST(c.customer_id AS STRING) = l.external_customer_id
LEFT JOIN vse_banka.digi_prodej.external_customer_service_address a
  ON CAST(a.address_id AS STRING) = l.external_address_id
LEFT JOIN active_marketing_consent mc
  ON mc.external_customer_id = l.external_customer_id;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_plan_digital_clients AS
SELECT
  c.client_id,
  CAST(c.created_at AS TIMESTAMP) AS created_at,
  c.status,
  LOWER(c.acquisition_channel) AS acquisition_channel,
  LOWER(c.onboarding_mode) AS onboarding_mode,
  CAST(c.mobile_app_user AS BOOLEAN) AS mobile_app_user,
  c.hashed_email,
  COALESCE(p.region, 'CZ') AS region,
  p.age_band AS age_band,
  p.birth_date,
  p.age_years,
  COALESCE(p.consent_any_marketing, FALSE) AS consent_any_marketing,
  COALESCE(p.consent_push, FALSE) AS consent_push,
  COALESCE(p.consent_email, FALSE) AS consent_email,
  COALESCE(p.consent_sms, FALSE) AS consent_sms,
  COALESCE(p.consent_phone, FALSE) AS consent_phone,
  c.dq_flag
FROM vse_banka.digi_prodej.bronze_plan_digital_clients c
LEFT JOIN vse_banka.digi_prodej.silver_customer_profile_enriched p
  ON c.client_id = p.client_id
 AND p.client_scope = 'plan'
WHERE c.dq_flag <> 'FAIL'
  AND c.status = 'active';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_plan_digital_accounts AS
SELECT
  account_id,
  client_id,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(activated_at AS TIMESTAMP) AS activated_at,
  status,
  LOWER(product_type) AS product_type,
  dq_flag
FROM vse_banka.digi_prodej.bronze_plan_digital_accounts
WHERE dq_flag <> 'FAIL'
  AND status = 'active'
  AND LOWER(product_type) = 'current_account';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_crm_clients AS
SELECT
  c.client_id,
  CAST(c.created_at AS TIMESTAMP) AS created_at,
  c.status,
  LOWER(c.acquisition_channel) AS acquisition_channel,
  LOWER(c.onboarding_mode) AS onboarding_mode,
  CAST(c.mobile_app_user AS BOOLEAN) AS mobile_app_user,
  c.hashed_email,
  COALESCE(p.region, 'CZ') AS region,
  p.age_band AS age_band,
  p.birth_date,
  p.age_years,
  COALESCE(p.consent_any_marketing, FALSE) AS consent_any_marketing,
  COALESCE(p.consent_push, FALSE) AS consent_push,
  COALESCE(p.consent_email, FALSE) AS consent_email,
  COALESCE(p.consent_sms, FALSE) AS consent_sms,
  COALESCE(p.consent_phone, FALSE) AS consent_phone,
  c.dq_flag
FROM vse_banka.digi_prodej.bronze_crm_clients c
LEFT JOIN vse_banka.digi_prodej.silver_customer_profile_enriched p
  ON c.client_id = p.client_id
 AND p.client_scope = 'current'
WHERE c.dq_flag <> 'FAIL'
  AND c.status = 'active';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_crm_accounts AS
SELECT
  account_id,
  client_id,
  CAST(created_at AS TIMESTAMP) AS created_at,
  CAST(activated_at AS TIMESTAMP) AS activated_at,
  status,
  LOWER(product_type) AS product_type,
  dq_flag
FROM vse_banka.digi_prodej.bronze_crm_accounts
WHERE dq_flag <> 'FAIL'
  AND status = 'active'
  AND LOWER(product_type) = 'current_account';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_transactions AS
SELECT
  transaction_id,
  client_id,
  CAST(date AS TIMESTAMP) AS date,
  CAST(amount AS DOUBLE) AS amount,
  CAST(mcc_code AS INT) AS mcc_code,
  merchant,
  dq_flag
FROM vse_banka.digi_prodej.bronze_transactions
WHERE dq_flag <> 'FAIL';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_ads_campaign_perf AS
SELECT
  DATE(date) AS date,
  LOWER(campaign_id) AS campaign,
  LOWER(platform) AS platform,
  SUM(cost_czk) AS total_cost_czk,
  SUM(clicks) AS clicks,
  SUM(impressions) AS impressions
FROM vse_banka.digi_prodej.bronze_ads_campaign_performance
WHERE dq_flag <> 'FAIL'
GROUP BY DATE(date), LOWER(campaign_id), LOWER(platform);

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_account_applications AS
SELECT
  c.conversion_id,
  c.user_id,
  c.session_id,
  c.event_time AS application_time,
  s.campaign,
  s.source AS platform,
  s.medium,
  s.device_type,
  s.region
FROM vse_banka.digi_prodej.silver_web_conversions c
JOIN vse_banka.digi_prodej.silver_web_sessions s
  ON c.session_id = s.session_id;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_team1_digi_income_cost_current_2025 AS
SELECT
  team_id,
  period,
  CAST(income_czk AS DOUBLE) AS income_czk,
  CAST(cost_czk AS DOUBLE) AS cost_czk,
  CAST(current_base_income_czk AS DOUBLE) AS current_base_income_czk,
  CAST(new_clients_income_czk AS DOUBLE) AS new_clients_income_czk,
  CAST(insurance_income_czk AS DOUBLE) AS insurance_income_czk,
  CAST(marketing_cost_czk AS DOUBLE) AS marketing_cost_czk,
  CAST(current_base_servicing_cost_czk AS DOUBLE) AS current_base_servicing_cost_czk,
  CAST(new_clients_servicing_cost_czk AS DOUBLE) AS new_clients_servicing_cost_czk,
  CAST(assisted_onboarding_cost_czk AS DOUBLE) AS assisted_onboarding_cost_czk,
  CAST(digital_platform_cost_czk AS DOUBLE) AS digital_platform_cost_czk,
  CAST(fte_cost_czk AS DOUBLE) AS fte_cost_czk,
  CAST(external_services_cost_czk AS DOUBLE) AS external_services_cost_czk,
  CAST(platform_support_cost_czk AS DOUBLE) AS platform_support_cost_czk,
  CAST(project_management_cost_czk AS DOUBLE) AS project_management_cost_czk,
  dq_flag
FROM vse_banka.digi_prodej.bronze_team1_digi_income_cost_current_2025
WHERE dq_flag <> 'FAIL';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_team1_digi_income_cost_2026 AS
SELECT
  team_id,
  period,
  CAST(income_czk AS DOUBLE) AS income_czk,
  CAST(cost_czk AS DOUBLE) AS cost_czk,
  CAST(current_base_income_czk AS DOUBLE) AS current_base_income_czk,
  CAST(new_clients_income_czk AS DOUBLE) AS new_clients_income_czk,
  CAST(insurance_income_czk AS DOUBLE) AS insurance_income_czk,
  CAST(marketing_cost_czk AS DOUBLE) AS marketing_cost_czk,
  CAST(current_base_servicing_cost_czk AS DOUBLE) AS current_base_servicing_cost_czk,
  CAST(new_clients_servicing_cost_czk AS DOUBLE) AS new_clients_servicing_cost_czk,
  CAST(assisted_onboarding_cost_czk AS DOUBLE) AS assisted_onboarding_cost_czk,
  CAST(digital_platform_cost_czk AS DOUBLE) AS digital_platform_cost_czk,
  CAST(fte_cost_czk AS DOUBLE) AS fte_cost_czk,
  CAST(external_services_cost_czk AS DOUBLE) AS external_services_cost_czk,
  CAST(platform_support_cost_czk AS DOUBLE) AS platform_support_cost_czk,
  CAST(project_management_cost_czk AS DOUBLE) AS project_management_cost_czk,
  dq_flag
FROM vse_banka.digi_prodej.bronze_team1_digi_income_cost_2026
WHERE dq_flag <> 'FAIL';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_plan_acquisition_monthly AS
SELECT
  month,
  CAST(cost_czk AS DOUBLE) AS cost_czk,
  CAST(sessions AS BIGINT) AS sessions,
  CAST(applications AS BIGINT) AS applications,
  CAST(digital_onboardings AS BIGINT) AS digital_onboardings,
  CAST(assisted_onboardings AS BIGINT) AS assisted_onboardings,
  CAST(new_clients_total AS BIGINT) AS new_clients_total,
  CAST(digital_share AS DOUBLE) AS digital_share,
  CAST(cpa_czk AS DOUBLE) AS cpa_czk
FROM vse_banka.digi_prodej.bronze_plan_acquisition_monthly;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_plan_insurance_targets AS
SELECT
  portfolio,
  product,
  CAST(target_policies AS BIGINT) AS target_policies,
  CAST(share_of_portfolio_target AS DOUBLE) AS share_of_portfolio_target,
  CAST(conversion_rate AS DOUBLE) AS conversion_rate,
  CAST(clients_to_contact AS DOUBLE) AS clients_to_contact
FROM vse_banka.digi_prodej.bronze_plan_insurance_targets;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.silver_plan_lending_targets AS
SELECT
  portfolio,
  product,
  CAST(target_volume_czk AS DOUBLE) AS target_volume_czk,
  CAST(share_of_volume AS DOUBLE) AS share_of_volume,
  CAST(share_of_clients AS DOUBLE) AS share_of_clients,
  CAST(avg_ticket_czk AS DOUBLE) AS avg_ticket_czk,
  warning
FROM vse_banka.digi_prodej.bronze_plan_lending_targets;

## Gold

### Gold - Acquisition / planning

In [0]:


CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_fact_acquisitions AS
SELECT
  uuid() AS record_id,
  DATE('2026-12-31') AS business_date,
  a.region,
  pa.client_id,
  DATE(acc.activated_at) AS date,
  a.campaign,
  a.platform,
  acc.activated_at,
  CASE
    WHEN a.region IS NOT NULL
      AND a.campaign IS NOT NULL
      AND a.platform IS NOT NULL
      AND pa.client_id IS NOT NULL
    THEN 'PASS'
    ELSE 'UNKNOWN'
  END AS dq_flag
FROM vse_banka.digi_prodej.silver_account_applications a
JOIN vse_banka.digi_prodej.silver_identity_map i
  ON a.user_id = i.user_id
JOIN vse_banka.digi_prodej.silver_plan_digital_clients pa
  ON i.client_id = pa.client_id
JOIN vse_banka.digi_prodej.silver_plan_digital_accounts acc
  ON pa.client_id = acc.client_id;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_mart_cpa_daily AS
WITH acquisition_counts AS (
  SELECT
    date,
    campaign,
    platform,
    region,
    COUNT(DISTINCT client_id) AS clients
  FROM vse_banka.digi_prodej.gold_fact_acquisitions
  GROUP BY date, campaign, platform, region
),
daily_costs AS (
  SELECT
    date,
    campaign,
    platform,
    total_cost_czk
  FROM vse_banka.digi_prodej.silver_ads_campaign_perf
)
SELECT
  uuid() AS record_id,
  DATE('2026-12-31') AS business_date,
  COALESCE(ac.region, 'CZ') AS region,
  dc.date,
  dc.campaign,
  dc.platform,
  COALESCE(ac.clients, 0) AS clients,
  dc.total_cost_czk,
  CASE
    WHEN COALESCE(ac.clients, 0) = 0 THEN NULL
    ELSE ROUND(dc.total_cost_czk / ac.clients, 2)
  END AS cpa_czk,
  COALESCE(MAX(fa.dq_flag), 'UNKNOWN') AS dq_flag
FROM daily_costs dc
LEFT JOIN acquisition_counts ac
  ON dc.date = ac.date
 AND dc.campaign = ac.campaign
 AND dc.platform = ac.platform
LEFT JOIN vse_banka.digi_prodej.gold_fact_acquisitions fa
  ON dc.date = fa.date
 AND dc.campaign = fa.campaign
 AND dc.platform = fa.platform
GROUP BY dc.date, dc.campaign, dc.platform, dc.total_cost_czk, ac.region, ac.clients;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_mart_cpa_monthly AS
SELECT
  uuid() AS record_id,
  DATE('2026-12-31') AS business_date,
  region,
  DATE_TRUNC('month', date) AS month,
  campaign,
  platform,
  SUM(clients) AS clients,
  SUM(total_cost_czk) AS total_cost_czk,
  CASE
    WHEN SUM(clients) = 0 THEN NULL
    ELSE ROUND(SUM(total_cost_czk) / SUM(clients), 2)
  END AS cpa_czk,
  MAX(dq_flag) AS dq_flag
FROM vse_banka.digi_prodej.gold_mart_cpa_daily
GROUP BY region, DATE_TRUNC('month', date), campaign, platform;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_mart_conversion_funnel_daily AS
SELECT
  month,
  sessions,
  applications AS leads,
  digital_onboardings AS clients,
  applications / sessions AS cr_session_to_lead,
  digital_onboardings / applications AS cr_lead_to_client,
  digital_onboardings / sessions AS cr_session_to_client
FROM vse_banka.digi_prodej.silver_plan_acquisition_monthly;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_plan_insurance_targets AS
SELECT * FROM vse_banka.digi_prodej.silver_plan_insurance_targets;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_plan_lending_targets AS
SELECT * FROM vse_banka.digi_prodej.silver_plan_lending_targets;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_team1_digi_income_cost_current_2025 AS
SELECT * FROM vse_banka.digi_prodej.silver_team1_digi_income_cost_current_2025;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_team1_digi_income_cost_2026 AS
SELECT * FROM vse_banka.digi_prodej.silver_team1_digi_income_cost_2026;

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_customer_profile_current_base AS
SELECT * FROM vse_banka.digi_prodej.silver_customer_profile_enriched
WHERE client_scope = 'current';

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_customer_profile_plan_base AS
SELECT * FROM vse_banka.digi_prodej.silver_customer_profile_enriched
WHERE client_scope = 'plan';


num_affected_rows,num_inserted_rows


### Gold - Vehicle Insurance Propensity features

In [0]:

CREATE OR REPLACE TABLE vse_banka.digi_prodej.gold_features_vehicle_propensity AS
WITH base_clients AS (
  SELECT client_id, region, age_band, birth_date, age_years
  FROM vse_banka.digi_prodej.silver_crm_clients
),
hist AS (
  SELECT
    client_id,
    date,
    amount,
    mcc_code,
    merchant,
    DATEDIFF(DATE('2025-12-31'), DATE(date)) AS days_from_snapshot
  FROM vse_banka.digi_prodej.silver_transactions
  WHERE DATE(date) <= DATE('2025-12-31')
),
agg_30 AS (
  SELECT
    client_id,
    SUM(CASE WHEN mcc_code IN (5541, 5542) THEN amount ELSE 0 END) AS fuel_spend_30d,
    SUM(CASE WHEN mcc_code IN (5541, 5542) THEN 1 ELSE 0 END) AS fuel_txn_count_30d
  FROM hist
  WHERE days_from_snapshot <= 30
  GROUP BY client_id
),
agg_90 AS (
  SELECT
    client_id,
    SUM(CASE WHEN mcc_code IN (5541, 5542) THEN amount ELSE 0 END) AS fuel_spend_90d,
    SUM(CASE WHEN mcc_code = 7538 THEN 1 ELSE 0 END) AS service_txn_count_90d,
    MAX(CASE WHEN merchant = 'dalnicni_znamka' THEN 1 ELSE 0 END) AS highway_vignette_flag,
    MAX(CASE WHEN merchant = 'insurance_company' THEN 1 ELSE 0 END) AS insurance_payment_flag,
    AVG(amount) AS avg_txn_amount,
    COUNT(*) AS total_txn_count_90d
  FROM hist
  WHERE days_from_snapshot <= 90
  GROUP BY client_id
),
last_insurance AS (
  SELECT
    client_id,
    MAX(DATE(date)) AS last_insurance_date
  FROM hist
  WHERE merchant = 'insurance_company'
  GROUP BY client_id
)
SELECT
  uuid() AS record_id,
  DATE('2025-12-31') AS business_date,
  b.region,
  b.client_id,
  DATE('2025-12-31') AS snapshot_date,
  COALESCE(a30.fuel_spend_30d, 0) AS fuel_spend_30d,
  COALESCE(a30.fuel_txn_count_30d, 0) AS fuel_txn_count_30d,
  COALESCE(a90.fuel_spend_90d, 0) AS fuel_spend_90d,
  COALESCE(a90.service_txn_count_90d, 0) AS service_txn_count_90d,
  COALESCE(a90.highway_vignette_flag, 0) AS highway_vignette_flag,
  COALESCE(a90.insurance_payment_flag, 0) AS insurance_payment_flag,
  CAST(DATEDIFF(DATE('2025-12-31'), li.last_insurance_date) AS INT) AS days_since_last_insurance,
  COALESCE(a90.avg_txn_amount, 0) AS avg_txn_amount,
  COALESCE(a90.total_txn_count_90d, 0) AS total_txn_count_90d,
  b.age_band,
  b.birth_date,
  b.age_years,
  'PASS' AS dq_flag
FROM base_clients b
LEFT JOIN agg_30 a30 ON b.client_id = a30.client_id
LEFT JOIN agg_90 a90 ON b.client_id = a90.client_id
LEFT JOIN last_insurance li ON b.client_id = li.client_id;

-- Final propensity scoring is produced only by the Python model step
-- in `Propensity model v5.py`.